# Agentic RAG: Building Smart Routing Agents

Standard RAG is a **linear pipeline**: Query -> Vector Search -> Generate. 
**Agentic RAG** introduces an LLM as a brain in the middle of the pipeline. The agent decides *if* it needs to search, *where* it should search, and *when* it has enough information to answer.

In this notebook, we'll build an Agentic Router that chooses between two tools: a simulated Vector Database (for company info) and a simulated Web Search (for real-world info).

We will build this using the two most popular agent frameworks:
1. **LangGraph** (by LangChain) - Uses a State Graph (nodes and edges).
2. **LlamaIndex Workflows** - Uses Event-Driven architecture.

In [ ]:
#!pip install langgraph langchain-community langchain-ollama llama-index-core llama-index-llms-ollama

## Part 1: LangGraph Implementation
LangGraph models agents as graphs. 
- **State**: A dictionary holding the conversation history.
- **Nodes**: Python functions (e.g., an LLM thinking, or a Tool executing).
- **Edges**: The logic that connects nodes (e.g., If the LLM requests a tool, go to the Tool Node).

In [1]:
import operator
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.tools import tool

# 1. Define our State. It just holds a list of messages that get appended together.
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

# 2. Define our simulated Tools
@tool
def vector_search(query: str) -> str:
    """Use this tool to search the vector database for company policies and documents."""
    print(f"  [TOOL] Executing Vector Search for: {query}")
    return "Company policy states that employees get 20 days of Paid Time Off (PTO)."

@tool
def web_search(query: str) -> str:
    """Use this tool to search the web for current events or weather."""
    print(f"  [TOOL] Executing Web Search for: {query}")
    return "The weather in New York is currently 72 degrees and sunny."

tools = [vector_search, web_search]
tool_mapping = {t.name: t for t in tools}

# 3. Setup our LLM and bind the tools to it so it knows they exist
llm = ChatOllama(model="qwen2.5:7b-instruct", temperature=0)
llm_with_tools = llm.bind_tools(tools)

c:\Users\sujat\projects\AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 4. Define our Graph Nodes
def agent_node(state):
    print("\n--- [NODE: Agent thinking...] ---")
    # Pass the conversation history to the LLM
    result = llm_with_tools.invoke(state["messages"])
    return {"messages": [result]}

def tool_node(state):
    print("--- [NODE: Executing Tool...] ---")
    last_message = state["messages"][-1]
    
    # Execute every tool the LLM asked for
    results = []
    for tool_call in last_message.tool_calls:
        tool_func = tool_mapping[tool_call["name"]]
        output = tool_func.invoke(tool_call["args"])
        # Store the tool result as a ToolMessage
        results.append(ToolMessage(content=str(output), tool_call_id=tool_call["id"], name=tool_call["name"]))
        
    return {"messages": results}

# 5. Define Conditional Routing Logic
def should_continue(state):
    last_message = state["messages"][-1]
    # If the LLM didn't call any tools, we are done!
    if not last_message.tool_calls:
        return "end"
    # Otherwise, go execute the tools
    return "continue"

# 6. Compile the Graph
workflow = StateGraph(AgentState)

workflow.add_node("agent", agent_node)
workflow.add_node("action", tool_node)

workflow.set_entry_point("agent")
workflow.add_conditional_edges(
    "agent", 
    should_continue, 
    {"continue": "action", "end": END}
)
workflow.add_edge("action", "agent") # After tool execution, always go back to the agent to synthesize

langgraph_app = workflow.compile()

In [4]:
# Let's test LangGraph!

queries = [
    "What is the weather in New York?",
    "How many days of PTO do I get?"
]

for query in queries:
    print(f"\n\n========== User Query: '{query}' ==========")
    final_state = langgraph_app.invoke({"messages": [HumanMessage(content=query)]})
    print("\n>>> Final Answer:")
    print(final_state["messages"][-1].content)



========== User Query: 'What is the weather in New York?' ==========

--- [NODE: Agent thinking...] ---
--- [NODE: Executing Tool...] ---
  [TOOL] Executing Web Search for: weather in New York

--- [NODE: Agent thinking...] ---

>>> Final Answer:
The current weather in New York is 72 degrees and it's sunny.


========== User Query: 'How many days of PTO do I get?' ==========

--- [NODE: Agent thinking...] ---
--- [NODE: Executing Tool...] ---
  [TOOL] Executing Vector Search for: PTO policy

--- [NODE: Agent thinking...] ---

>>> Final Answer:
According to our company's policy, you are entitled to 20 days of Paid Time Off (PTO). If you need any further details or have additional questions, feel free to ask!


## Part 2: LlamaIndex Workflows Implementation
LlamaIndex Workflows are **Event-Driven**. Instead of pre-wiring nodes together, functions emit specific Events, and other functions are triggered by those Events. This allows for highly flexible and decoupled architectures.

In [8]:
from llama_index.core.workflow import (
    StartEvent, StopEvent, Workflow, step, Event
)
from llama_index.llms.ollama import Ollama
import asyncio

# 1. Define our Custom Event
class NeedToolEvent(Event):
    tool_name: str
    query: str

# Setup LLM
llama_llm = Ollama(model="qwen2.5:7b-instruct", request_timeout=120.0)

# 2. Define the Workflow Class
class AgenticRouterWorkflow(Workflow):
    
    @step
    async def router_step(self, ev: StartEvent) -> NeedToolEvent | StopEvent:
        print("\n--- [LlamaIndex: Router Step thinking...] ---")
        user_query = ev.get("query")
        
        # For this educational example, we use a simple prompt to act as the router.
        # In production, you would use function calling similar to the LangGraph example.
        prompt = f"""
        Decide which tool to use for this query:
        - 'vector' if the query asks about company policies or PTO.
        - 'web' if the query asks about weather or news.
        - 'none' if it's a general greeting.
        
        Query: {user_query}
        Reply with ONLY the tool name ('vector', 'web', or 'none').
        """
        
        decision = str(await llama_llm.acomplete(prompt)).strip().lower()
        
        if "vector" in decision:
            print("  -> Emitting NeedToolEvent for: vector")
            return NeedToolEvent(tool_name="vector", query=user_query)
        elif "web" in decision:
            print("  -> Emitting NeedToolEvent for: web")
            return NeedToolEvent(tool_name="web", query=user_query)
        else:
            print("  -> No tools needed, emitting StopEvent")
            return StopEvent(result="I don't need tools to answer that.")
            
    @step
    async def tool_executor_step(self, ev: NeedToolEvent) -> StopEvent:
        print("--- [LlamaIndex: Tool Executor Step running...] ---")
        
        if ev.tool_name == "vector":
            print(f"  [TOOL] Simulating vector search for: {ev.query}")
            context = "Company policy states that employees get 20 days of Paid Time Off (PTO)."
        else:
            print(f"  [TOOL] Simulating web search for: {ev.query}")
            context = "The weather in New York is currently 72 degrees and sunny."
            
        # Synthesize the final answer using the context
        final_prompt = f"Context: {context}\n\nAnswer the user query: {ev.query}"
        print("--- [LlamaIndex: Synthesizing Final Answer...] ---")
        final_answer = str(await llama_llm.acomplete(final_prompt))
        
        return StopEvent(result=final_answer)

In [9]:
# Run LlamaIndex Workflow
async def run_llama_workflow():
    workflow = AgenticRouterWorkflow(timeout=120, verbose=False)
    
    for query in queries:
        print(f"\n\n========== User Query: '{query}' ==========")
        result = await workflow.run(query=query)
        print("\n>>> Final Answer:")
        print(result)

# In Jupyter, we can directly await async functions
await run_llama_workflow()



========== User Query: 'What is the weather in New York?' ==========

--- [LlamaIndex: Router Step thinking...] ---
  -> Emitting NeedToolEvent for: web
--- [LlamaIndex: Tool Executor Step running...] ---
  [TOOL] Simulating web search for: What is the weather in New York?
--- [LlamaIndex: Synthesizing Final Answer...] ---

>>> Final Answer:
The current weather in New York is 72 degrees with sunny conditions.


========== User Query: 'How many days of PTO do I get?' ==========

--- [LlamaIndex: Router Step thinking...] ---
  -> Emitting NeedToolEvent for: vector
--- [LlamaIndex: Tool Executor Step running...] ---
  [TOOL] Simulating vector search for: How many days of PTO do I get?
--- [LlamaIndex: Synthesizing Final Answer...] ---

>>> Final Answer:
According to company policy, you get 20 days of Paid Time Off (PTO).


LangChain (linear):
  Input → LLM → Tool → LLM → Output

LangGraph (graph with loops/branches):
         ┌──────────────┐
  Input →│     Agent    │→ END
         └──────┬───────┘
                │ (needs a tool)
         ┌──────▼───────┐
         │  Tool Call   │
         └──────┬───────┘
                │ (loop back)
         └──────────────┘

#LangChain is an open-source framework for building applications powered by large language models (LLMs). It provides tools and abstractions to make it easier to connect LLMs with external data, tools, and workflows.
Core Idea
LLMs alone are stateless and can only work with what's in their context window. LangChain helps you build chains of logic that combine LLMs with memory, tools, data sources, and other components to create more powerful applications.
Key Components
Chains — Sequences of calls (to LLMs, tools, or other components) linked together. For example: take user input → search a database → pass results to an LLM → return a response.
Prompts — Reusable prompt templates with dynamic variables, so you can structure inputs to LLMs consistently.
Models — Wrappers around LLMs (OpenAI, Anthropic, Hugging Face, etc.) and embedding models, giving them a unified interface.
Memory — Mechanisms to persist context across multiple turns in a conversation, since LLMs are stateless by default.
Retrieval / RAG — Tools to fetch relevant documents from vector stores (like Pinecone, Chroma, FAISS) and inject them into prompts. This is the backbone of Retrieval-Augmented Generation (RAG).
Agents — LLMs that can decide which tools to use and in what order to accomplish a goal. For example, an agent might decide to search the web, then do a calculation, then write a summary.
Tools — Functions the agent can call: web search, calculators, APIs, code execution, etc.

User question
     ↓
Retrieve relevant docs from vector DB
     ↓
Inject docs into prompt template
     ↓
Send to LLM
     ↓
Return answer to user

LangGraph is a framework built on top of LangChain for creating stateful, multi-step AI agents and workflows modeled as graphs.
The Core Idea
Regular LangChain chains are linear (A → B → C). But real-world agent logic often needs:

Loops (retry until something works)
Branching (if X do this, else do that)
Multiple agents talking to each other
Persistent state across many steps

LangGraph solves this by letting you define your workflow as a directed graph where nodes are functions/agents and edges are the connections between them.
Key Concepts
Nodes — Individual units of work. Each node is a Python function that takes the current state and returns an updated state. A node could be an LLM call, a tool call, or any custom logic.
Edges — Connections between nodes. They can be:

Fixed — always go from node A to node B
Conditional — based on the current state, decide which node to go to next

State — A shared object (usually a dictionary) that gets passed between nodes and updated at each step. This is what makes it stateful.
Graph — The overall structure that wires nodes and edges together.